In [3]:
import os
import xml.etree.ElementTree as ET

# Paths
xml_folder = "data/raw/annotations"
yolo_folder = "data/processed/labels"

os.makedirs(yolo_folder, exist_ok=True)

# Class mapping
class_map = {
    "With Helmet": 0,
    "Without Helmet": 1
}


def convert_box(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]

    x_center = (box[0] + box[1]) / 2.0
    y_center = (box[2] + box[3]) / 2.0

    w = box[1] - box[0]
    h = box[3] - box[2]

    x_center *= dw
    w *= dw
    y_center *= dh
    h *= dh

    return (x_center, y_center, w, h)


for xml_file in os.listdir(xml_folder):

    if not xml_file.endswith(".xml"):
        continue

    xml_path = os.path.join(xml_folder, xml_file)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)

    txt_filename = xml_file.replace(".xml", ".txt")
    txt_path = os.path.join(yolo_folder, txt_filename)

    with open(txt_path, "w") as out_file:

        for obj in root.findall("object"):

            cls_name = obj.find("name").text.strip()

            if cls_name not in class_map:
                continue

            cls_id = class_map[cls_name]

            xml_box = obj.find("bndbox")

            xmin = float(xml_box.find("xmin").text)
            xmax = float(xml_box.find("xmax").text)
            ymin = float(xml_box.find("ymin").text)
            ymax = float(xml_box.find("ymax").text)

            bbox = convert_box((w, h), (xmin, xmax, ymin, ymax))

            out_file.write(
                str(cls_id) + " " +
                " ".join([str(round(a, 6)) for a in bbox]) +
                "\n"
            )

print("Conversion completed!")

Conversion completed!


In [4]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Paths
image_dir = "data/processed/images"
label_dir = "data/processed/labels"

# Get all images
images = [f for f in os.listdir(image_dir) if f.endswith(".jpg") or f.endswith(".png")]

# Split dataset
train, test = train_test_split(images, test_size=0.1, random_state=42)
train, val = train_test_split(train, test_size=0.2, random_state=42)

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))


# Create folders
for split in ["train", "val", "test"]:
    
    os.makedirs(f"{image_dir}/{split}", exist_ok=True)
    os.makedirs(f"{label_dir}/{split}", exist_ok=True)


def move_files(file_list, split):

    for file in file_list:

        image_src = os.path.join(image_dir, file)
        label_src = os.path.join(label_dir, file.replace(".jpg", ".txt").replace(".png",".txt"))

        image_dst = os.path.join(image_dir, split, file)
        label_dst = os.path.join(label_dir, split, file.replace(".jpg",".txt").replace(".png",".txt"))

        if os.path.exists(image_src):
            shutil.move(image_src, image_dst)

        if os.path.exists(label_src):
            shutil.move(label_src, label_dst)


# Move files
move_files(train, "train")
move_files(val, "val")
move_files(test, "test")

print("Dataset successfully split!")

Train: 549
Validation: 138
Test: 77
Dataset successfully split!


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="data/dataset.yaml",
    epochs=1
)

ModuleNotFoundError: No module named 'ultralytics'